# COMP7103 Lab 9 · Compare two clusterings of news headlines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab9/Lab9_news_clusters.ipynb)

You have 1,000 news headlines from 2014 and want to know what topics they cover, without reading them all. You cluster them in two ways, **TF-IDF + K-means** and **embeddings + HDBSCAN**, describe each cluster in words (c-TF-IDF keywords, and a name written by an LLM, a large language model), and then **check the labels** by reading members of the clusters.

| Time | What you do |
|---|---|
| 5 min | Brief |
| 30 min | Guided work: cluster, label, audit |
| 10 min | Share-out |

This lab is ungraded practice for the final exam. Data: News Aggregator (Gasparetti, UCI Machine Learning Repository), CC BY 4.0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the headlines, their embeddings and their UMAP coordinates, all computed in advance.

In [ ]:
import os, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab9/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 9 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 9 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab9")
    return os.path.join("data", name)

df = pd.read_csv(get("headlines.csv"))
headlines = df["headline"]
emb = np.load(get("emb.npy"))          # 1,000 x 384 sentence embeddings
umap5 = np.load(get("umap5.npy"))      # 1,000 x 5 UMAP coordinates (HDBSCAN clusters these)
umap2 = np.load(get("umap2.npy"))      # 1,000 x 2 UMAP coordinates (for the picture only)
print(len(df), "headlines · embeddings", emb.shape, "· UMAP", umap5.shape)

## 1 · Look at the headlines (2 min)

1,000 English headlines, one from each of 1,000 different news stories. The `category` column is the news site's own label (business, science & technology, entertainment, health; 250 each). The clusterings do not use it; we look at it only at the end.

In [ ]:
df.sample(8, random_state=1)

## 2 · Pipeline 1: TF-IDF + K-means (6 min)

TF-IDF (Lecture 2) turns each headline into a vector with one weight per word: high when the word is in the headline and rare in the collection. We drop English stop words (*the*, *of*, …) and keep only words that appear in **at least 2** headlines.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=___)
X = tfidf.fit_transform(headlines)
print("TF-IDF matrix:", X.shape, "(headlines x words)")

K-means needs k. **Rule:** try k = 5, 10, 15, 20 and keep the k with the highest silhouette. The silhouette (Part 5) is computed in the TF-IDF space with **cosine distance** (1 − cosine similarity), the usual distance for text vectors. Each K-means run keeps the best of 10 starts (`n_init=10`). This cell takes about 10 seconds.

In [ ]:
sil_km = {}                                # k -> silhouette
for k in [5, 10, 15, 20]:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = kmeans.fit_predict(X)          # one cluster number per headline
    sil_km[k] = silhouette_score(X, labels, metric=___)
    print("k =", k, "· silhouette", round(sil_km[k], 3))

best_k = 5                                  # the k with the highest silhouette
for k in sil_km:
    if sil_km[k] > sil_km[best_k]:
        best_k = k
print("best k =", best_k)

In [ ]:
kmeans = KMeans(n_clusters=___, n_init=10, random_state=0)
km_labels = kmeans.fit_predict(X)
km_result = {"clusters": best_k, "noise %": 0.0, "silhouette": round(sil_km[best_k], 3)}
print(km_result)
sizes = pd.Series(km_labels).value_counts()     # the number of headlines in each cluster, largest first
print("cluster sizes:", list(sizes))

K-means puts every headline in some cluster, so its noise share is 0%. Look at the sizes: what is special about the largest cluster? A silhouette near 0 means a headline is about as close to the next cluster as to its own. The silhouette keeps rising slowly with k and stays near 0; we stop at 20 so that the clusters stay readable.

## 3 · Pipeline 2: embeddings → UMAP → HDBSCAN (6 min)

Each headline was embedded in advance with the sentence-embedding model all-MiniLM-L6-v2 (384 numbers per headline, Lecture 2). UMAP then squeezed the 384 dimensions to 5, keeping near neighbours near (settings: `n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=7103`). UMAP gives slightly different coordinates on different machines, so we use the shipped ones. The optional cell below recomputes them; leave it off for the rest of the lab, or your clusters will not match the prepared LLM message.

In [ ]:
RECOMPUTE_UMAP = False      # optional: True recomputes the 5-d coordinates (about 30 s; results differ a little)
if RECOMPUTE_UMAP:
    import umap
    reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=7103)
    umap5 = reducer.fit_transform(emb)

HDBSCAN is a density-based method like DBSCAN (Part 4). It tries many density levels and keeps the clusters that last longest, so there is no k and no radius to choose. Its main setting is `min_cluster_size`, the smallest group it may call a cluster. For 1,000 headlines we use **10** (1% of the data). Headlines in no dense group get the label −1: **noise**.

In [ ]:
hdbscan = HDBSCAN(min_cluster_size=___)
hd_labels = hdbscan.fit_predict(umap5)  # one cluster number per headline; -1 = noise
cluster_numbers = set(hd_labels)        # the different labels
cluster_numbers.discard(-1)             # -1 (noise) is not a cluster
n_hd = len(cluster_numbers)
noise = ___                             # share of headlines labelled -1
print(n_hd, "clusters · noise share", round(noise, 3))

The silhouette of HDBSCAN is computed in the **5-d UMAP space** (the space it clustered in) with Euclidean distance, **noise points left out** (they belong to no cluster).

In [ ]:
keep = ___                              # True for clustered headlines (label not -1)
sil_hd = silhouette_score(umap5[keep], hd_labels[keep])
hd_result = {"clusters": n_hd, "noise %": round(100 * noise, 1), "silhouette": round(sil_hd, 3)}
results = pd.DataFrame({"TF-IDF + K-means": km_result, "Embeddings + HDBSCAN": hd_result}).T   # one row per pipeline
results["clusters"] = results["clusters"].astype(int)     # show 20, not 20.0
results

The two silhouettes are measured in different spaces, so they are not a fair comparison: UMAP packs neighbours tightly on purpose. One fairer check is to measure both clusterings in the **same** space, the 384-d embeddings with cosine distance (HDBSCAN still without its noise points, which flatters it a little).

In [ ]:
sil_km_emb = silhouette_score(emb, km_labels, metric="cosine")
sil_hd_emb = silhouette_score(emb[keep], hd_labels[keep], metric="cosine")     # noise left out, as before
print("Silhouette in the 384-d embedding space (cosine):")
print("  TF-IDF + K-means:    ", round(sil_km_emb, 3))
print("  Embeddings + HDBSCAN:", round(sil_hd_emb, 3))

A picture: a separate UMAP run to 2 dimensions, coloured by each clustering (grey = HDBSCAN noise).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))      # two pictures side by side
names = ["TF-IDF + K-means", "Embeddings + HDBSCAN"]
all_labels = [km_labels, hd_labels]
for i in range(2):
    ax = axes[i]
    labels = all_labels[i]
    noise = labels == -1                    # True for noise headlines (HDBSCAN only)
    clustered = labels != -1
    x = umap2[:, 0]                         # the 2-D UMAP coordinates
    y = umap2[:, 1]
    ax.scatter(x[noise], y[noise], s=6, c="lightgrey", label="noise")
    # one colour per cluster; with more than 20 clusters the 20 colours are reused (% 20)
    ax.scatter(x[clustered], y[clustered], s=8, c=labels[clustered] % 20, cmap="tab20")
    n_clusters = len(set(labels[clustered]))
    ax.set_title(f"{names[i]}: {n_clusters} clusters")
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(loc="lower right", markerscale=2)
plt.suptitle("1,000 headlines in 2-D (UMAP); one colour per cluster")
plt.show()

## 4 · Describe the clusters with c-TF-IDF (4 min)

Class-based TF-IDF joins all headlines of a cluster into one document and asks which words are frequent in this cluster but rare across clusters:

$$W_{t,c} = \mathrm{tf}_{t,c} \times \log\left(1 + \frac{A}{f_t}\right)$$

$\mathrm{tf}_{t,c}$: share of the words in cluster $c$ that are term $t$. $f_t$: count of $t$ over all clusters. $A$: average number of words per cluster. Stop words and numbers are dropped (also *doesn* from *doesn't*), and HDBSCAN's noise is left out. Fill in $A$ and $W$; the top 6 words of each cluster are its keywords. With few headlines per cluster many words tie; here ties are broken alphabetically, so the last keywords of a small cluster can be arbitrary (the lecture broke ties by closeness to the cluster's average embedding).

In [ ]:
def ctfidf(labels, top=6):
    """Size and top c-TF-IDF keywords of every cluster (noise, -1, left out)."""
    clustered = labels != -1                    # leave out the noise
    # Count words: no stop words, only words of 2+ letters (the pattern), and not "doesn" from "doesn't"
    cv = CountVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b(?!'t)")
    counts = cv.fit_transform(headlines[clustered]).toarray()     # headlines x words
    words = cv.get_feature_names_out()          # the word of each column
    cluster_of = labels[clustered]              # the cluster of each counted headline
    clusters = np.unique(cluster_of)            # the cluster numbers, sorted

    # M[c, t] = how often word t appears in cluster c (all its headlines joined into one document)
    M = []
    for k in clusters:
        M.append(counts[cluster_of == k].sum(axis=0))
    M = np.array(M)                             # clusters x words

    tf = np.zeros(M.shape)
    for c in range(len(clusters)):
        tf[c] = M[c] / M[c].sum()               # share of the words in cluster c that are term t
    f = M.sum(axis=0)                           # count of t over all clusters
    A = ___                                     # average number of words per cluster (total word count / number of clusters)
    W = ___

    rows = []
    for c in range(len(clusters)):
        order = np.argsort(-W[c], kind="stable")    # word positions, highest W first (equal W: A to Z)
        keywords = words[order[:top]]               # the top words
        size = (labels == clusters[c]).sum()        # headlines in the cluster
        rows.append({"size": size, "keywords": ", ".join(keywords)})
    return pd.DataFrame(rows, index=clusters)

kw_km = ctfidf(km_labels)
kw_hd = ctfidf(hd_labels)
kw_km.sort_values("size", ascending=False)

In [ ]:
kw_hd.sort_values("size", ascending=False)

Which keyword lists read like a topic, and which look like a bag of unrelated words?

## 5 · Ask an LLM to name 6 clusters (3 min)

We name and audit 3 clusters of each pipeline. **Rule:** of the clusters with at least 20 members (10 for the LLM to read, the rest to draw your 5 audit headlines from), take the largest, the middle one by size, and the smallest. Run the next cell to see which clusters are groups A–F.

In [ ]:
def audit_choice(labels):
    """Of the clusters with at least 20 members, the largest, the middle one and the smallest."""
    sizes = pd.Series(labels[labels != -1]).value_counts()     # cluster number -> number of headlines
    big = sizes[sizes >= 20]
    # largest first; equal sizes: the smaller cluster number first
    big = big.sort_index().sort_values(ascending=False, kind="stable")
    order = list(big.index)
    return [order[0], order[len(order) // 2], order[-1]]

LABELS = {"TF-IDF + K-means": km_labels, "Embeddings + HDBSCAN": hd_labels}
GROUPS = {}                                   # letter -> (pipeline, cluster number)
for pipe in LABELS:
    for c in audit_choice(LABELS[pipe]):
        letter = "ABCDEF"[len(GROUPS)]        # A, B, C for K-means, then D, E, F for HDBSCAN
        GROUPS[letter] = (pipe, c)

def members(letter):
    """The 10 headlines the LLM sees, and 5 other members of the same cluster for your audit."""
    pipe = GROUPS[letter][0]
    c = GROUPS[letter][1]
    positions = np.flatnonzero(LABELS[pipe] == c)      # the row numbers of the headlines in this cluster
    rng = np.random.default_rng(7103 + "ABCDEF".index(letter))    # a fixed random order for each group
    shuffled = rng.permutation(positions)
    return shuffled[:10], shuffled[10:15]

for g in GROUPS:
    pipe = GROUPS[g][0]
    c = GROUPS[g][1]
    size = (LABELS[pipe] == c).sum()
    print(f"Group {g}: {pipe}, cluster {c} ({size} headlines)")

1. Run the next cell. It prints a message with 10 headlines from each group and checks that your groups match it.
2. Paste the message into any chat LLM you use. The headlines are public news; the message holds no personal data.
3. Paste the LLM's whole answer between the triple quotes in the cell after.

No LLM available? Use the ready-made answer (see the comment in that cell). It comes from Qwen2.5-1.5B-Instruct, a small open model.

In [ ]:
prompt = open(get("llm_prompt.txt")).read()
print(prompt)
same = True                                  # are all your LLM headlines in the prepared message?
for g in GROUPS:
    for i in members(g)[0]:
        if headlines[i] not in prompt:
            same = False
if same:
    print("Your groups match the prepared message.")
else:
    print("WARNING: your clusters differ from the prepared message. Check k, min_cluster_size and RECOMPUTE_UMAP.")

In [ ]:
llm_reply = """
paste the LLM's answer here
"""
# No LLM? Use the ready-made answer instead:
# llm_reply = open(get("llm_fallback.txt")).read()
print(llm_reply)

This cell picks the names out of the answer. If a name is missing or wrong, type it in by hand, for example `llm_names["A"] = "Tech deals"`.

In [ ]:
llm_names = {}                                # group letter -> the name the LLM gave
for line in llm_reply.splitlines():
    start = line.find("Group")
    if start == -1:
        continue                              # this line names no group
    rest = line[start + len("Group"):].strip()     # e.g. "A: Tech deals" or "B** - Health news"
    if rest == "" or rest[0] not in "ABCDEF":
        continue
    letter = rest[0]
    k = 1
    while k < len(rest) and rest[k] in " *)|":
        k = k + 1                             # skip spaces, bold marks and brackets after the letter
    if k == len(rest) or rest[k] not in ":.-–—":
        continue                              # no ":" (or "." or a dash) between the letter and the name
    name = rest[k + 1:].strip(" *")
    if name != "":
        llm_names[letter] = name
llm_names

## 6 · Audit the labels by hand (9 min)

**Coverage** is the share of a cluster's members that a label really describes. For each group you see both labels and 5 members the LLM did **not** see. For each headline, ask: would a reader who sees only this label expect this headline in the group?

- The **keywords** describe a headline when it is about a topic the keywords name (a mixed list may name several topics); a word shared by chance (*care* in *eye-care*, *new*, *says*) does not count.
- The **LLM name** describes a headline when the headline is about what the name says. A name so vague that it fits any news (*Current events*) describes nothing: count 0.

Decide quickly: if you hesitate for more than a few seconds, count the headline as not described.

*Worked example* (group C, ready-made LLM name *Health issues and public health initiatives*; two members you will not audit):

- *South Korea's state health insurer sues Philip Morris, BAT for smoking damages*: keywords yes (health); LLM name yes (harm from smoking, a public health action).
- *Ackman, Valeant want to create eye-care, skin-drug giant*: keywords no (*care* only by chance in *eye-care*; it is a business deal); LLM name no (a company merger, not a health issue).

Count, for each group, how many of the 5 headlines each label describes.

In [ ]:
kw = {"TF-IDF + K-means": kw_km, "Embeddings + HDBSCAN": kw_hd}
for g in GROUPS:
    pipe = GROUPS[g][0]
    k = GROUPS[g][1]
    size = (LABELS[pipe] == k).sum()
    print(f"Group {g} · {pipe}, cluster {k} ({size} headlines)")
    print("  c-TF-IDF keywords:", kw[pipe].loc[k, "keywords"])
    print("  LLM name:         ", llm_names.get(g, "(missing)"))     # .get: "(missing)" if there is no name
    audit = members(g)[1]                 # the 5 headlines for your audit
    for n in range(5):
        print(f"  {n + 1:2d}. {headlines[audit[n]]}")
    print()

In [ ]:
counts = {     # group: (headlines the c-TF-IDF keywords describe, headlines the LLM name describes), out of 5
    "A": (___, ___),
    "B": (___, ___),
    "C": (___, ___),
    "D": (___, ___),
    "E": (___, ___),
    "F": (___, ___),
}
# One row per group; counts out of 5 become percentages
cov = pd.DataFrame(counts, index=["c-TF-IDF coverage %", "LLM coverage %"]).T * 100 / 5
cov.insert(0, "pipeline", [GROUPS[g][0] for g in cov.index])            # add a column in front
cov.insert(1, "LLM name", [llm_names.get(g, "") for g in cov.index])
cov

Mean coverage over the 3 audited groups of each pipeline, next to the clustering numbers: this is your row for the class board.

In [ ]:
# The mean coverage of the 3 groups of each pipeline
mean_cov = cov.groupby("pipeline")[["c-TF-IDF coverage %", "LLM coverage %"]].mean().round(0)
board = results.join(mean_cov)            # add the two columns to the results table, matched by pipeline
board

**After the fact.** The news site's own categories in each audited cluster (all its members). Does a cluster that mixes categories also have lower coverage?

In [ ]:
categories = {}                           # group -> number of headlines per news-site category
for g in GROUPS:
    pipe = GROUPS[g][0]
    k = GROUPS[g][1]
    in_cluster = LABELS[pipe] == k
    categories[g] = df["category"][in_cluster].value_counts()
# One row per group; a category missing from a cluster gets 0
pd.DataFrame(categories).T.fillna(0).astype(int)

## 7 · Share-out (10 min)

- Report your row for each pipeline: number of clusters, noise share (%), silhouette, and the mean c-TF-IDF and LLM coverage (%) over your 3 audited groups; we collect the class's typical values on the board.
- Which pipeline has the higher silhouette? Say in which space each was computed; is the comparison fair?
- Find one label that sounds right but describes only part of its cluster. Was it the keywords or the LLM name?
- The habit from the lecture: check a cluster label against the members, not against how well it reads.